# MeaningFlow: Coverage Gap Analysis Demo

This notebook walks through the core MeaningFlow pipeline:

1. Load a demand corpus (user queries) and a supply corpus (existing content)
2. Build semantic graphs for both
3. Find coverage gaps — topics with high demand and no content
4. Visualize the clusters and gaps

No external data files needed — sample data is embedded below.

## Setup

```bash
pip install meaningflow[viz]
```

Or if running from a cloned repo:

```bash
pip install -e ".[viz]"
```

In [ ]:
from meaningflow import SemanticGraph
import pandas as pd
import numpy as np

## Sample Data

We'll simulate a pet supplies website. The **demand** corpus represents what users are searching for. The **supply** corpus represents the content pages that already exist on the site.

The supply intentionally has strong dog and cat coverage, but gaps in reptiles, birds, fish, pet insurance, and pet travel.

In [ ]:
# --- DEMAND: what users are searching for ---
queries = [
    # Dog cluster (well covered)
    "best dog food for puppies",
    "how to train a puppy not to bite",
    "puppy training schedule by week",
    "best dry dog food brands",
    "grain free dog food pros and cons",
    "how to crate train a puppy",
    "puppy potty training tips",
    "best dog toys for aggressive chewers",
    "how to stop a dog from barking",
    "best dog beds for large breeds",
    "raw diet for dogs benefits",
    "how much should I feed my puppy",
    "best leash for dogs that pull",
    "dog dental chews that actually work",
    "best dog shampoo for itchy skin",
    "how to trim dog nails safely",
    "best dog crates for separation anxiety",
    "how to socialize a puppy",
    "homemade dog food recipes vet approved",
    "best dog harness no pull",
    # Cat cluster (well covered)
    "best cat litter for odor control",
    "how to stop a cat from scratching furniture",
    "best wet cat food for indoor cats",
    "cat litter box training tips",
    "best cat trees for large cats",
    "how to introduce two cats",
    "cat food for sensitive stomach",
    "best automatic cat feeder",
    "how to brush a cat that hates it",
    "best cat toys for bored indoor cats",
    "clumping vs non clumping cat litter",
    "kitten food vs adult cat food",
    "how often should you clean a litter box",
    "best cat carrier for vet visits",
    "how to get a cat to drink more water",
    "cat anxiety symptoms and treatment",
    "stressed cat signs",
    "cat hiding under bed all day",
    "nervous cat behavior",
    "calming treats for cats",
    # Reptile cluster (GAP - no supply)
    "best terrarium for bearded dragon",
    "bearded dragon care guide for beginners",
    "leopard gecko habitat setup",
    "best heat lamp for reptiles",
    "ball python enclosure requirements",
    "reptile UVB lighting guide",
    "best substrate for ball pythons",
    "how to set up a gecko tank",
    "reptile thermostat for heat mat",
    "corn snake care sheet",
    "bearded dragon diet vegetables",
    "crested gecko food list",
    "reptile misting system setup",
    "how to handle a new ball python",
    "best reptile fogger for humidity",
    # Bird cluster (GAP - no supply)
    "best bird cage for parakeets",
    "how to teach a budgie to talk",
    "cockatiel care guide for beginners",
    "best bird food for cockatiels",
    "parrot toys for mental stimulation",
    "how to clip bird wings safely",
    "signs of a sick parakeet",
    "best bird bath for small birds",
    "canary singing tips and training",
    "how to hand tame a budgie",
    "bird cage placement in house",
    "best pellet food for parrots",
    # Fish cluster (GAP - no supply)
    "best freshwater fish for beginners",
    "how to cycle a fish tank",
    "betta fish tank setup guide",
    "best aquarium filter for 20 gallon tank",
    "live plants for freshwater aquarium",
    "how to lower ammonia in fish tank",
    "best fish tank heater",
    "tropical fish compatible with bettas",
    "aquarium water testing kit guide",
    "how to clean a fish tank properly",
    "best LED light for planted aquarium",
    "shrimp tank setup for beginners",
    # Pet insurance cluster (GAP - no supply)
    "best pet insurance for dogs",
    "pet insurance worth it or not",
    "cheapest pet insurance plans",
    "pet insurance that covers pre existing conditions",
    "how much does pet insurance cost per month",
    "pet insurance vs vet savings account",
    "emergency vet bill help",
    "pet health plan comparison",
    "best pet insurance for older dogs",
    "cat insurance plans ranked",
    # Pet travel cluster (GAP - no supply)
    "how to travel with a dog on a plane",
    "best airline approved pet carrier",
    "dog car sickness remedies",
    "pet friendly hotels near me",
    "camping with dogs tips",
    "how to road trip with a cat",
    "pet passport requirements",
    "best dog car seat for small dogs",
    "anxiety medication for dogs during travel",
    "international pet travel requirements",
]

# Simulated search volumes (higher = more demand)
query_volumes = (
    [200, 350, 180, 400, 150, 300, 500, 250, 280, 220,
     120, 380, 190, 160, 210, 170, 140, 260, 130, 310]  # dogs
    + [300, 280, 350, 200, 180, 220, 250, 190, 160, 210,
       270, 230, 320, 140, 180, 150, 120, 90, 110, 160]  # cats
    + [180, 220, 160, 250, 200, 140, 170, 130, 120, 150,
       190, 110, 100, 130, 90]  # reptiles
    + [150, 120, 180, 140, 110, 90, 100, 80, 70, 130, 95, 160]  # birds
    + [200, 250, 180, 220, 160, 190, 170, 210, 140, 230, 150, 120]  # fish
    + [300, 250, 350, 200, 280, 220, 180, 260, 310, 150]  # pet insurance
    + [220, 280, 150, 200, 170, 120, 140, 190, 160, 130]  # pet travel
)

print(f"Demand corpus: {len(queries)} queries")
print(f"Total search volume: {sum(query_volumes):,}")

In [ ]:
# --- SUPPLY: content pages that already exist ---
content = [
    # Dog content (strong coverage)
    "The Complete Puppy Training Guide: Week by Week Schedule",
    "Best Dry Dog Food Brands Reviewed and Ranked",
    "Grain-Free Dog Food: What You Need to Know",
    "Crate Training Your Puppy: A Step-by-Step Guide",
    "Potty Training a Puppy: Timeline and Tips",
    "10 Best Dog Toys for Heavy Chewers",
    "How to Stop Excessive Barking: Trainer-Approved Methods",
    "Best Dog Beds for Large and Giant Breeds",
    "Raw Food Diet for Dogs: Benefits, Risks, and Recipes",
    "Puppy Feeding Guide: How Much and How Often",
    "Best No-Pull Dog Harnesses Reviewed",
    "Dog Dental Health: Best Chews and Treats",
    "Best Dog Shampoos for Sensitive and Itchy Skin",
    "How to Trim Your Dog's Nails Without Stress",
    "Puppy Socialization Guide: When, Where, and How",
    # Cat content (strong coverage)
    "Best Cat Litter for Odor Control: Top Picks Reviewed",
    "How to Stop Your Cat from Scratching Furniture",
    "Best Wet Cat Food for Indoor Cats",
    "Litter Box Training: A Complete Guide for New Cat Owners",
    "Best Cat Trees and Towers for Large Cats",
    "Introducing a New Cat to Your Household",
    "Best Cat Food for Sensitive Stomachs",
    "Automatic Cat Feeders: Buyer's Guide",
    "Cat Grooming Tips: How to Brush a Reluctant Cat",
    "Best Interactive Cat Toys for Indoor Entertainment",
    "Clumping vs Non-Clumping Litter: Which Is Better?",
    "Kitten Nutrition Guide: Choosing the Right Food",
    "Litter Box Maintenance: How Often to Clean and Replace",
    "Best Cat Carriers for Stress-Free Vet Visits",
    "How to Get Your Cat to Drink More Water",
    # Intentionally NO content for: reptiles, birds, fish, pet insurance, pet travel
]

print(f"Supply corpus: {len(content)} pages")

## Build Semantic Graphs

Fit a `SemanticGraph` for both demand (queries) and supply (content). The first run will download the Sentence-BERT model (~90MB).

In [ ]:
demand = SemanticGraph(
    texts=queries,
    volumes=query_volumes,
    embedder="all-MiniLM-L6-v2",
    min_cluster_size=5,      # small for this demo dataset
    min_samples=3,
    umap_n_neighbors=10,
    umap_n_components=5,
    random_state=42,
)
demand.fit()

print(f"Demand: {demand.n_clusters} clusters, {demand.noise_ratio:.1%} noise")

In [ ]:
supply = SemanticGraph(
    texts=content,
    embedder="all-MiniLM-L6-v2",
    min_cluster_size=5,
    min_samples=3,
    umap_n_neighbors=8,
    umap_n_components=5,
    random_state=42,
)
supply.fit()

print(f"Supply: {supply.n_clusters} clusters, {supply.noise_ratio:.1%} noise")

## Inspect Clusters

Let's see what topics the pipeline discovered.

In [ ]:
print("=" * 70)
print("DEMAND CLUSTERS (what users are searching for)")
print("=" * 70)
for c in demand.clusters:
    print(f"\nCluster {c.id} (n={c.size}, volume={c.volume:,})")
    for term in c.top_terms[:5]:
        print(f"  - {term}")

In [ ]:
print("=" * 70)
print("SUPPLY CLUSTERS (content that already exists)")
print("=" * 70)
for c in supply.clusters:
    print(f"\nCluster {c.id} (n={c.size})")
    for term in c.top_terms[:5]:
        print(f"  - {term}")

## Find Coverage Gaps

This is the core output: demand clusters that have no matching supply cluster above the similarity threshold.

In [ ]:
gaps = demand.coverage_gaps(reference=supply, similarity_threshold=0.55)

print(f"Found {len(gaps)} coverage gaps out of {demand.n_clusters} demand clusters")
print("=" * 70)

for gap in gaps:
    print(f"\nGAP: Cluster {gap.id} (size={gap.size}, volume={gap.volume:,})")
    print(f"  Nearest supply: '{gap.nearest_supply}' (similarity={gap.nearest_similarity:.3f})")
    print(f"  Top queries:")
    for term in gap.top_terms[:5]:
        print(f"    - {term}")

## Visualize: Demand Clusters

Project the demand embeddings to 2D and color by cluster. Gap clusters are highlighted in red.

In [ ]:
import matplotlib.pyplot as plt
import umap

# Reduce to 2D for visualization
reducer_2d = umap.UMAP(
    n_neighbors=10, n_components=2, metric="cosine", random_state=42
)
coords_2d = reducer_2d.fit_transform(demand.embeddings)

# Identify which cluster IDs are gaps
gap_ids = {g.id for g in gaps}

fig, ax = plt.subplots(figsize=(12, 8))
fig.patch.set_facecolor("#0B1220")
ax.set_facecolor("#0B1220")

# Plot noise points
noise_mask = demand.labels == -1
if noise_mask.any():
    ax.scatter(
        coords_2d[noise_mask, 0], coords_2d[noise_mask, 1],
        s=15, color="#3A4459", alpha=0.4, label="Noise"
    )

# Plot covered clusters
colors_covered = ["#6FD3F7", "#B9A1FF", "#5DCEA8", "#6FD3F7", "#B9A1FF"]
colors_gap = ["#F5B041", "#F07DB1", "#FF6B6B", "#FFD93D", "#FF8C42"]

covered_idx = 0
gap_idx = 0

for cluster in demand.clusters:
    mask = demand.labels == cluster.id
    is_gap = cluster.id in gap_ids
    
    if is_gap:
        color = colors_gap[gap_idx % len(colors_gap)]
        gap_idx += 1
        marker = "^"
        size = 60
        edge = "white"
    else:
        color = colors_covered[covered_idx % len(colors_covered)]
        covered_idx += 1
        marker = "o"
        size = 40
        edge = "none"
    
    label_text = cluster.top_terms[0][:30] if cluster.top_terms else f"Cluster {cluster.id}"
    if is_gap:
        label_text = f"GAP: {label_text}"
    
    ax.scatter(
        coords_2d[mask, 0], coords_2d[mask, 1],
        s=size, color=color, marker=marker, alpha=0.85,
        edgecolors=edge, linewidths=0.5, label=label_text
    )

ax.legend(
    loc="upper left", fontsize=8, framealpha=0.3,
    facecolor="#131B2E", edgecolor="#3A4459",
    labelcolor="#E8EAF0"
)
ax.set_title(
    "Demand Clusters: Covered vs Gaps",
    color="#E8EAF0", fontsize=16, pad=15
)
ax.tick_params(colors="#6B7891")
for spine in ax.spines.values():
    spine.set_color("#3A4459")

plt.tight_layout()
plt.show()

## Summary Table

A clean table of gap clusters ranked by volume — the editorial team's content roadmap.

In [ ]:
gap_data = []
for g in gaps:
    gap_data.append({
        "Topic": ", ".join(g.top_terms[:3]),
        "Queries": g.size,
        "Volume": g.volume,
        "Nearest Supply": g.nearest_supply[:40],
        "Similarity": f"{g.nearest_similarity:.3f}",
    })

gap_df = pd.DataFrame(gap_data)
gap_df.index = range(1, len(gap_df) + 1)
gap_df.index.name = "Priority"
gap_df

## Graph Structure

MeaningFlow also builds a NetworkX graph over the clusters. Let's inspect it.

In [ ]:
G = demand.graph
print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(f"\nEdges (inter-cluster relationships):")
for u, v, data in G.edges(data=True):
    u_terms = demand.get_cluster(u).top_terms[0][:25] if demand.get_cluster(u) else str(u)
    v_terms = demand.get_cluster(v).top_terms[0][:25] if demand.get_cluster(v) else str(v)
    print(f"  {u_terms} <-> {v_terms}  (similarity={data.get('weight', 0):.3f})")

## Next Steps

In a production workflow, the gap table above feeds directly into your editorial process:

1. **Taxonomy expansion** — each gap cluster is a candidate for a new category branch
2. **Content briefs** — the top terms in each gap become the seed keywords for new content
3. **Synonym discovery** — terms within the same cluster that use different surface forms are synonym candidates
4. **Monthly monitoring** — re-run this analysis monthly to catch emerging demand and track whether gaps are being filled

For more detail on how MeaningFlow fits into a full knowledge engineering stack, see:
- [Knowledge Engineering for Search and Content](https://medium.com/@brian-curry-research)
- [Building a Knowledge Engineering System](https://medium.com/@brian-curry-research)